In [4]:
from pathlib import Path
import os


def _find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / ".git").exists():
            return candidate
    return start


def load_env_file() -> None:
    repo_root = _find_repo_root(Path.cwd())
    env_path = repo_root / ".env"
    example_path = repo_root / ".env.example"
    target = env_path if env_path.exists() else example_path
    if not target.exists():
        raise FileNotFoundError(
            f"Expected either {env_path} or {example_path} to exist."
        )

    with target.open() as handle:
        for raw_line in handle:
            line = raw_line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, value = line.split("=", 1)
            os.environ.setdefault(key, value)

    print(f"Loaded environment variables from {target.relative_to(repo_root)}")


load_env_file()


Loaded environment variables from .env


In [5]:
from langchain.agents import create_agent

In [6]:
def get_weather(city: str) -> str:
    """Get weather for a given city"""
    return f"Its always sunny in {city}"

agent = create_agent(
    model = 'openai:gpt-4o-mini',
    tools = [get_weather],
    prompt = 'You are a helpful assistant',
)

In [7]:
agent.invoke(
    {"message": [{"role":"user", "content": "what is the weather in sf"}]}
)

{'messages': [AIMessage(content='How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 46, 'total_tokens': 54, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_560af6e559', 'id': 'chatcmpl-CNY22Tt1gW4354cR3SlJj5Lmbi8z0', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--4bb06a8e-a357-494d-98d3-62b5cb85550b-0', usage_metadata={'input_tokens': 46, 'output_tokens': 8, 'total_tokens': 54, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})]}

## Weather Forecasting Agent

In [8]:
# Step 1: System Prompt - The Agent's initial instructions or personality.
system_prompt = """You are an expert weather forecaster, who speaks in puns.

You have access to two tools:

- get_weather_for_location: use this to get the weather for a specific location
- get_user_location: use this to get the user's location

If a user asks you for the weather, make sure you know the location. If you can tell from the question that they mean whereever they are, use the get_user_location tool to find their location."""

In [9]:
# Step 2: Create tools - tools are functions that can be called, they interact with external data to get stuff done.
from langchain_core.tools import tool
import random

def get_weather_for_location(city: str) -> str:
    '''Get weather for a given city'''
    conditions = random.choice(['sunny', 'rainy', 'cloudy'])
    return f'It is {conditions} in {city}'

from langchain_core.runnables import RunnableConfig

# A lookup table for demo purposes
USER_LOCATION = {
    "1":"Florida",
    "2":"SF"
}

'''
@tool decorator turns Python callables into LangChain `StructuredTool` objects
that the agent can discover and invoke. It can then use LangChain's tool metadata 
like names, descriptions, config injections.
'''
@tool
def get_user_location(config: RunnableConfig) -> str:
    '''Retrieve user information'''
    user_id = config.get("configurable", {}).get("user_id")
    return USER_LOCATION[user_id]


In [10]:
# Step 3: Configure the model
from langchain.chat_models import init_chat_model

model = init_chat_model(
    "openai:gpt-4o-mini",
    temperature=0,
)

In [11]:
# Step 4: Define response format
from dataclasses import dataclass

@dataclass
class WeatherResponse:
    conditions: str
    punny_response: str

In [12]:
# Step 5: Add memory for the agent to remember conversation history
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()

In [13]:
# Step 6: Bring it all together
agent = create_agent(
    model=model,
    prompt=system_prompt,
    tools=[get_user_location, get_weather_for_location],
    response_format=WeatherResponse,
    checkpointer=checkpointer
)

# config = {"configurable": {"thread_id": "1"}}
# context = {"user_id": "1"}

'''
`config` is the run metadata shared across every runnable (models, tools, graphs).
`config` has reserved keys like "configurable", "run_name", "tags", "metadata", "callbacks".
"configurable" is a catch-all for values we want to read back inside the graph or tools.
"thread_id" must be supplied when using `InMemorySaver` or any checkpointer - it decides which conversation thread to load.
'''

config = {"configurable": {"thread_id": "1", "user_id": "2"}}

response = agent.invoke(
    {"messages": [{"role": "user", "content": "what is the weather outside?"}]},
    config=config,
)

response['structured_response']

WeatherResponse(conditions='cloudy', punny_response="Looks like the clouds are having a little party in the sky! Don't forget your umbrella, just in case they decide to rain on your parade!")

**More control over the model using provider's package**

```python
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model="gpt-5",
    temperature=0.1,
    max_tokens=1000,
    timeout=30
)
agent = create_agent(model, tools=tools)
```

In [15]:

response = agent.invoke(
    {"messages": [{"role": "user", "content": "thank you!"}]},
    config=config
)

response['structured_response']

WeatherResponse(conditions='thankful', punny_response="You're welcome! I'm always here to brighten your day, even when the weather's a bit cloudy!")

# More About Agents

In [20]:
# Dynamic Model Loading
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent, AgentState
from langgraph.runtime import Runtime

def select_model(state: AgentState, runtime: Runtime) -> ChatOpenAI:
    '''Chooses model based on conversation complexity'''
    messages = state['messages']
    message_count = len(messages)
    print(messages)

    if message_count < 10:
        return ChatOpenAI(model='gpt-4.1-mini').bind_tools(tools)
    else:
        return ChatOpenAI(model='gpt-5').bind_tools(tools)

In [27]:
# ToolNode: Creating tools like before (callables, @tool, provider's dict) will internally create
# a ToolNode. Here, you can defined ToolNode yourself for finer control.
from langchain_core.tools import tool
from langchain.agents import ToolNode
from langchain.agents import create_agent

tool_node = ToolNode(
    tools = [get_user_location, get_weather_for_location],
    handle_tool_errors = 'check again if error'
)

agent = create_agent(model, tools=tool_node)
result = agent.invoke({"messages":[{"role": "user", "content": "what is the weather outside?"}]})
result

{'messages': [HumanMessage(content='what is the weather outside?', additional_kwargs={}, response_metadata={}, id='3c2f9d6f-4a73-4895-99e8-d4f070ff0b49'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_ru7ggc53Kok1PqGAAhu4GZva', 'function': {'arguments': '{}', 'name': 'get_user_location'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 11, 'prompt_tokens': 65, 'total_tokens': 76, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_95d112f245', 'id': 'chatcmpl-CNZuEHVw6KwTvQrHMHSgbwKgIOuH8', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--9aaab395-9bdb-43e4-933e-5f120a2474b2-0', tool_calls=[{'name': 'get_user_location', 'args': {}, 'id': 'call_ru7ggc53Kok1PqGAA

In [26]:
model = ChatOpenAI(
    model='gpt-4.1-mini'
)